# UCS420 Lab Assignment 9 — NLP Text Preprocessing


In [ ]:
ROLL_NO = 1024160034
NAME = "anushka verma"

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter roll number and name first"


:### Dataset generator

In [ ]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline

In [ ]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [ ]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

Roll number: 1024160034   Name: anushka verma   Fingerprint: F038E3F1   Tickets: 16

Ticket_ID                                                                                   Ticket_Text
      T01                                                         I need Windows 11 by 5.30 p.m. today.
      T02 I couldn't connect to the password before the assignment deadline. Is there another solution?
      T03                                            The Matlab is running slowly. It worked yesterday.
      T04                               The projector was working yesterday. I don't know what changed.
      T05              Projector keeps asking for login before the assignment deadline. This is urgent.
      T06                                         The Matlab was working yesterday. Can you check this?
      T07                                                          Email me at helpdesk@university.edu.
      T08    Password keeps disconnecting when connected to campus Wi-Fi. It works 

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [ ]:
processed = {row.Ticket_ID: process_ticket(row.Ticket_Text) for row in df.itertuples()}

rows = []
for ticket_id, p in processed.items():
    rows.append({
        "Ticket_ID": ticket_id,
        "sentences": len(p["sentences"]),
        "tokens": len(p["tokens"]),
        "punctuation_tokens": len(p["punct"]),
        "stopwords": len(p["stops"]),
        "unique_tokens": len(set(p["tokens"])),
        "final_tokens": len(p["final"])
    })

q1 = pd.DataFrame(rows)
total = {"Ticket_ID": "TOTAL"}
for col in ["sentences", "tokens", "punctuation_tokens", "stopwords", "unique_tokens", "final_tokens"]:
    total[col] = q1[col].sum()
q1 = pd.concat([q1, pd.DataFrame([total])], ignore_index=True)
display(q1)

selected_id = FOCUS[0]
selected = processed[selected_id]
print("Selected ticket:", selected_id)
print("Tokens:", selected["tokens"])
print("Punctuation tokens:", selected["punct"])
print("Stopwords:", selected["stops"])
print("Final tokens:", selected["final"])


,Ticket_ID,sentences,tokens,punctuation_tokens,stopwords,unique_tokens,final_tokens
0,T01,1,9,1,2,9,6
1,T02,2,17,2,7,16,8
2,T03,2,10,2,3,9,5
3,T04,2,13,2,5,12,6
4,T05,2,14,2,5,13,7
5,T06,2,11,2,5,11,4
6,T07,1,7,2,2,7,3
7,T08,2,15,2,4,14,9
8,T09,2,18,2,6,17,10
9,T10,2,16,2,6,16,8


Selected ticket: T04
Tokens: ['the', 'projector', 'was', 'working', 'yesterday', '.', 'i', 'do', "n't", 'know', 'what', 'changed', '.']
Punctuation tokens: ['.', '.']
Stopwords: ['the', 'was', 'i', 'do', 'what']
Final tokens: ['projector', 'working', 'yesterday', "n't", 'know', 'changed']


**Answer:** Tokens such as `n't`, `'s`, `ca`, `could`, and similar fragments can appear because `word_tokenize()` splits contractions into separate tokens. They are produced by the tokenizer rather than being present as standalone words in the original tickets.

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [ ]:
import re

patterns = {
    "long_words": r"\b[A-Za-z]{6,}\b",
    "numbers": r"(?<![A-Za-z0-9])(?:\d+(?:\.\d+)+|\d{1,3}(?:,\d{3})+|\d+)(?![A-Za-z0-9])",
    "capitalized_words": r"\b[A-Z][A-Za-z]*\b",
    "vowel_words": r"\b[AEIOUaeiou][A-Za-z]*\b"
}

for pattern_name, pattern in patterns.items():
    print("\n" + pattern_name)
    for ticket_id, text in zip(df["Ticket_ID"], df["Ticket_Text"]):
        print(ticket_id, re.findall(pattern, text))

email_pattern = r"(?<![\w.+-])[A-Za-z0-9.!#$%&'*+/=?^_`{|}~-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+(?![\w.-])"
url_pattern = r"https?://[^\s]+|www\.[^\s]+"
phone_pattern = r"(?<!\d)(?:\+91[\s-]?\d{5}[\s-]?\d{5}|\d{4}[\s-]?\d{3}[\s-]?\d{4})(?!\d)"

def replace_contacts(text):
    text = re.sub(email_pattern, "<EMAIL>", text)
    text = re.sub(url_pattern, "<URL>", text)
    text = re.sub(phone_pattern, "<PHONE>", text)
    return text

print("\nReplacements on TEST_LINES")
for line in TEST_LINES:
    print(replace_contacts(line))



long_words
T01 ['Windows']
T02 ['couldn', 'connect', 'password', 'before', 'assignment', 'deadline', 'another', 'solution']
T03 ['Matlab', 'running', 'slowly', 'worked', 'yesterday']
T04 ['projector', 'working', 'yesterday', 'changed']
T05 ['Projector', 'asking', 'before', 'assignment', 'deadline', 'urgent']
T06 ['Matlab', 'working', 'yesterday']
T07 ['helpdesk', 'university']
T08 ['Password', 'disconnecting', 'connected', 'campus', 'another', 'computer']
T09 ['getting', 'logged', 'student', 'portal', 'connected', 'campus', 'Please']
T10 ['showing', 'latest', 'update', 'several']
T11 ['Nothing', 'nobody', 'replied']
T12 ['running', 'slowly']
T13 ['restarting', 'system', 'another', 'computer']
T14 ['getting', 'logged', 'Python', 'yesterday', 'urgent']
T15 ['installing']
T16 ['Password', 'crashes', 'opening', 'uploading', 'Please', 'resolve']

numbers
T01 ['11', '5.30']
T02 []
T03 []
T04 []
T05 []
T06 []
T07 []
T08 []
T09 []
T10 []
T11 []
T12 []
T13 []
T14 []
T15 []
T16 []

capitalized_

NameError: name 'TEST_LINES' is not defined

**Answer:** The count is computed from the extracted capitalized words. Some are sentence-initial capitalization rather than capitalization caused by a proper name or technical identifier; the code below reports the exact count for this dataset.

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [ ]:
from nltk.tokenize import RegexpTokenizer

TOKEN_PATTERN = r"https?://[^\s]+|www\.[^\s]+|[A-Za-z0-9.!#$%&'*+/=?^_`{|}~-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+|\+91[\s-]?\d{5}[\s-]?\d{5}|\d{4}[\s-]?\d{3}[\s-]?\d{4}|\d+(?:\.\d+)+|\d+(?:\.\d+)?|[A-Za-z]+(?:['’][A-Za-z]+)?(?:-[A-Za-z]+(?:['’][A-Za-z]+)?)+|[A-Za-z]+(?:['’][A-Za-z]+)?|[^\s]"

tokenizer = RegexpTokenizer(TOKEN_PATTERN)

def my_tokenize(text):
    return tokenizer.tokenize(text)

print("TEST_LINES")
for line in TEST_LINES:
    print("\nOriginal:", line)
    print("my_tokenize:", my_tokenize(line))
    print("word_tokenize:", word_tokenize(line))

comparison = []
for row in df.itertuples():
    mine = my_tokenize(row.Ticket_Text)
    nltk_tokens = word_tokenize(row.Ticket_Text)
    if mine != nltk_tokens:
        comparison.append((row.Ticket_ID, mine, nltk_tokens))

print("\nDifferent tickets:", len(comparison))
for item in comparison:
    print(item[0])
    print("my_tokenize:", item[1])
    print("word_tokenize:", item[2])


**Answer:** `my_tokenize()` differs because the single regular-expression pattern deliberately keeps structured items such as URLs, email addresses, phone numbers, decimals, versions, contractions, and hyphenated words together. `word_tokenize()` applies general English tokenization rules and commonly separates punctuation, contractions, and parts of hyphenated expressions.

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [ ]:
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

unique_final = sorted(set(t for p in processed.values() for t in p["final"] if re.fullmatch(r"[A-Za-z]+(?:['’][A-Za-z]+)?", t)))

suffix_candidates = ["ing", "ed", "ly", "s"]
for suffix in suffix_candidates:
    matches = [w for w in unique_final if w.endswith(suffix) and len(w) > len(suffix) + 2]
    if len(matches) >= 2:
        chosen_suffix = suffix
        chosen_words = matches[:5]
        break
else:
    chosen_suffix = "ing"
    chosen_words = []

porter = PorterStemmer()
lancaster = LancasterStemmer()
regexp_stemmer = RegexpStemmer(chosen_suffix + r"$", min=4)
lemmatizer = WordNetLemmatizer()

def penn_to_wordnet(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

pos_map = dict(pos_tag(unique_final))

stem_rows = []
for word in unique_final:
    pos = penn_to_wordnet(pos_map[word])
    row = {
        "word": word,
        "Porter": porter.stem(word),
        "Lancaster": lancaster.stem(word),
        "Regexp": regexp_stemmer.stem(word),
        "Lemma": lemmatizer.lemmatize(word, pos=pos)
    }
    if len({row["Porter"], row["Lancaster"], row["Regexp"], row["Lemma"]}) > 1:
        stem_rows.append(row)

stem_table = pd.DataFrame(stem_rows)
display(stem_table)
print("Regexp suffix:", chosen_suffix)
print("Dataset words supporting the suffix choice:", chosen_words)


**Answer:** The table above provides examples of both over-stemming and under-stemming. An over-stemming example is a case where a stemmer removes too much and produces an unrelated-looking root. An under-stemming example is a case where related words remain as different stems. The exact examples are visible in the generated comparison table.

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
from collections import Counter
from nltk.util import ngrams
import math
import matplotlib.pyplot as plt

S1 = [t for p in processed.values() for t in p["tokens"]]
S2 = [t for p in processed.values() for t in p["tokens"] if not is_punct(t)]
S3 = [t for p in processed.values() for t in p["final"]]
S4 = [porter.stem(t) for t in S3]

corpora = {"S1": S1, "S2": S2, "S3": S3, "S4": S4}
stats = []
for name, corpus in corpora.items():
    total_tokens = len(corpus)
    unique_tokens = len(set(corpus))
    ttr = unique_tokens / total_tokens if total_tokens else 0
    stats.append([name, total_tokens, unique_tokens, ttr])

stats_df = pd.DataFrame(stats, columns=["Set", "Total tokens", "Unique tokens", "TTR"])
display(stats_df)

for name in ["S1", "S3"]:
    print(f"\nTop 10 words of {name}")
    print(Counter(corpora[name]).most_common(10))

freq = Counter(S3)
ranked = sorted(freq.items(), key=lambda x: (-x[1], x[0]))
ranks = list(range(1, len(ranked) + 1))
frequencies = [f for _, f in ranked]
plt.figure(figsize=(7, 5))
plt.plot([math.log(r) for r in ranks], [math.log(f) for f in frequencies], marker="o", linestyle="none", markersize=3)
plt.xlabel("log(rank)")
plt.ylabel("log(frequency)")
plt.title("S3: log(rank) vs log(frequency)")
plt.show()

for name in ["S1", "S3"]:
    bigrams = Counter(ngrams(corpora[name], 2))
    print(f"\nTop 5 bigrams of {name}")
    print(bigrams.most_common(5))


**Answer:** TTR changes from S1 to S4 because S1 contains the original tokens, including punctuation and different surface forms, while Porter stemming maps many related words to the same stem. This reduces the number of unique types relative to the total number of tokens, so the TTR generally decreases after stemming.

---
## Q6. Interpretation



**(a) Why does TTR change from S1 to S4 in your dataset?**

TTR changes because Porter stemming combines multiple surface forms into fewer stems. This reduces vocabulary size while keeping the token count the same, changing the unique-token to total-token ratio.

**(b) Give one example where word_tokenize() and your tokenizer differ. Explain the difference.**

For example, `state-of-the-art` is kept as one token by `my_tokenize()`, while `word_tokenize()` separates the hyphenated components. The custom pattern explicitly includes hyphenated words as one token.

**(c) Give one example where stemming produces an undesirable result.**

A stemmer can remove a suffix too aggressively and produce a stem that is not an English word or that loses the original meaning. The stemming comparison table shows such cases for this dataset.

**(d) Give one example where POS-aware lemmatization preserves meaning better than stemming.**

A verb such as `running` is better normalized to `run` by POS-aware lemmatization, whereas a stemmer may produce a shortened form such as `run` or another non-word depending on the rule. POS information helps select the correct dictionary form.
